# Notebook 2 — Basic GANs

**Phase 2** deliverable. Learning objectives:
- Vanilla GAN architecture
- DCGAN with best practices
- Conditional GAN for controlled generation
- Detecting & addressing mode collapse
- Wasserstein GAN for training stability

Complete the TODOs in `models/gans/` and `training/losses.py` (adversarial,
WGAN, and gradient-penalty losses) before running the training cells. Run
`pytest -m gan -v` to check your progress.


In [ ]:
import sys
from pathlib import Path

# Make `generative_art_studio` importable without `pip install -e .`
REPO_ROOT = Path.cwd().parent if (Path.cwd() / "notebooks").exists() else Path.cwd()
sys.path.insert(0, str(REPO_ROOT / "src"))

import torch
from generative_art_studio.utils import set_seed, plot_image_grid
from generative_art_studio.config import DEVICE

set_seed(42)
print(f"Using device: {DEVICE}")


In [ ]:
from generative_art_studio.data import SyntheticImageDataset, get_dataloader

dataset = SyntheticImageDataset(num_samples=256, image_size=64, num_classes=2)
dataloader = get_dataloader(dataset, batch_size=32)


## 1. Vanilla GAN (fully-connected) — reference architecture

In [ ]:
from generative_art_studio.models.gans import VanillaGenerator, VanillaDiscriminator
from generative_art_studio.training.train_gan import train_gan
from torch.optim import Adam

latent_dim = 64
generator = VanillaGenerator(latent_dim=latent_dim).to(DEVICE)
discriminator = VanillaDiscriminator().to(DEVICE)

g_opt = Adam(generator.parameters(), lr=2e-4, betas=(0.5, 0.999))
d_opt = Adam(discriminator.parameters(), lr=2e-4, betas=(0.5, 0.999))

history = train_gan(generator, discriminator, dataloader, g_opt, d_opt, latent_dim, device=DEVICE, epochs=3)


In [ ]:
import matplotlib.pyplot as plt
plt.plot(history.g_loss, label="G loss")
plt.plot(history.d_loss, label="D loss")
plt.legend(); plt.title("Vanilla GAN training — watch for D loss collapsing near 0 (mode collapse warning sign)")
plt.show()

from generative_art_studio.utils.latent_space import sample_latent
z = sample_latent(16, latent_dim, DEVICE)
samples = generator(z)
plot_image_grid(samples.detach().cpu(), nrow=4, title="Vanilla GAN samples")


## 2. DCGAN

Implement `DCGANGenerator`/`DCGANDiscriminator` in `models/gans/dcgan.py`,
then repeat the training loop above with these models (remember to call
`.apply(weights_init_dcgan)` right after construction) and `latent_dim=100`.


In [ ]:
from generative_art_studio.models.gans import DCGANGenerator, DCGANDiscriminator, weights_init_dcgan

dc_latent_dim = 100
dc_generator = DCGANGenerator(latent_dim=dc_latent_dim, feature_maps=32).to(DEVICE)
dc_discriminator = DCGANDiscriminator(feature_maps=32).to(DEVICE)
dc_generator.apply(weights_init_dcgan)
dc_discriminator.apply(weights_init_dcgan)

# TODO: train dc_generator/dc_discriminator the same way as the vanilla GAN above,
# then visualize samples and compare training-curve stability.


## 3. Conditional GAN

Implement `ConditionalGenerator`/`ConditionalDiscriminator` in
`models/gans/conditional_gan.py`, then generate samples per class label to
demonstrate controlled generation.


In [ ]:
from generative_art_studio.models.gans import ConditionalGenerator, ConditionalDiscriminator

cgan_gen = ConditionalGenerator(latent_dim=64, num_classes=2).to(DEVICE)

# TODO: train, then compare generator output for label=0 vs label=1 with the same z
# z = sample_latent(8, 64, DEVICE)
# samples_label0 = cgan_gen(z, torch.zeros(8, dtype=torch.long, device=DEVICE))
# samples_label1 = cgan_gen(z, torch.ones(8, dtype=torch.long, device=DEVICE))


## 4. Wasserstein GAN (WGAN-GP)

Implement `WGANCritic` (`models/gans/wgan.py`) and `wgan_critic_loss` /
`wgan_generator_loss` / `gradient_penalty` (`training/losses.py`), then
write your own critic/generator training loop below — unlike the BCE GAN,
WGAN trains the critic `N_CRITIC` steps per generator step (see
`config.N_CRITIC`).


In [ ]:
from generative_art_studio.models.gans import WGANCritic
from generative_art_studio.training.losses import wgan_critic_loss, wgan_generator_loss, gradient_penalty
from generative_art_studio import config

wgan_gen = DCGANGenerator(latent_dim=100, feature_maps=32).to(DEVICE)  # generator arch is unchanged
critic = WGANCritic(feature_maps=32).to(DEVICE)

# TODO: implement the WGAN-GP training loop:
#   for N_CRITIC steps: update critic using wgan_critic_loss + WGAN_GP_LAMBDA * gradient_penalty
#   then: one generator update using wgan_generator_loss
# Compare training-curve smoothness against the vanilla/DCGAN BCE-based runs above.


## Reflection (for your Technical Report)

- Did you observe **mode collapse** in any model (e.g. all generated
  samples looking nearly identical)? Show a sample grid as evidence.
- Compare G/D loss curve *shape* across vanilla GAN, DCGAN, and WGAN — which
  was most stable, and why (tie back to what each loss function optimizes)?
- What did conditioning on class label change about controllability?
